# 🚦 YOLOv8 Extra Large - Traffic Sign Detection Training
**Colab Pro | A100 GPU | Mapillary Traffic Sign Dataset**

This notebook trains YOLOv8x (Extra Large) at 1280px resolution for absolute MAXIMUM accuracy.

### Steps:
1. Mount Google Drive
2. Upload and extract dataset
3. Install dependencies
4. Train YOLOv8x
5. Download best weights

## ⚙️ Step 1: Check GPU & Mount Drive

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project directory
import os
PROJECT_DIR = '/content/drive/MyDrive/traffic-sign-training'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"\n📁 Project dir: {PROJECT_DIR}")

## 📦 Step 2: Upload & Extract Dataset

**Upload your `yolo_dataset.zip` file to Google Drive** at:
```
My Drive/traffic-sign-training/yolo_dataset.zip
```

Or upload it directly to this Colab session using the file browser on the left sidebar.

In [ ]:
import os
import zipfile

# Check if dataset is in Drive
DRIVE_ZIP = os.path.join(PROJECT_DIR, 'yolo_dataset.zip')
LOCAL_ZIP = '/content/yolo_dataset.zip'

if os.path.exists(DRIVE_ZIP):
    print(f"✅ Found dataset in Drive: {DRIVE_ZIP}")
    zip_path = DRIVE_ZIP
elif os.path.exists(LOCAL_ZIP):
    print(f"✅ Found dataset uploaded locally: {LOCAL_ZIP}")
    zip_path = LOCAL_ZIP
else:
    # Upload from local machine
    print("📤 No zip found. Upload it now...")
    from google.colab import files
    uploaded = files.upload()
    zip_path = LOCAL_ZIP

# Extract dataset
DATASET_DIR = '/content/yolo_dataset'
if not os.path.exists(DATASET_DIR):
    print(f"📦 Extracting dataset to {DATASET_DIR}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print("✅ Extraction complete!")
else:
    print("✅ Dataset already extracted.")

# Show structure
!ls -la /content/yolo_dataset/
!echo "---"
!ls /content/yolo_dataset/images/ 2>/dev/null || ls /content/yolo_dataset/yolo/images/ 2>/dev/null

In [ ]:
import yaml
import glob

# Find and fix data.yaml paths for Colab
yaml_candidates = glob.glob('/content/yolo_dataset/**/data.yaml', recursive=True)
if not yaml_candidates:
    raise FileNotFoundError("data.yaml not found! Check extraction.")

YAML_PATH = yaml_candidates[0]
print(f"📋 Found: {YAML_PATH}")

# Read and fix paths
with open(YAML_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the base path to point to Colab location
data_dir = os.path.dirname(YAML_PATH)
data_config['path'] = data_dir

# Write updated yaml
COLAB_YAML = '/content/data.yaml'
with open(COLAB_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"\n✅ Updated data.yaml saved to {COLAB_YAML}")
print(f"   Classes: {data_config['nc']}")
print(f"   Path: {data_config['path']}")
print(f"   Train: {data_config['train']}")
print(f"   Val: {data_config['val']}")

# Verify images exist
train_dir = os.path.join(data_dir, data_config['train'])
val_dir = os.path.join(data_dir, data_config['val'])
train_count = len(os.listdir(train_dir)) if os.path.exists(train_dir) else 0
val_count = len(os.listdir(val_dir)) if os.path.exists(val_dir) else 0
print(f"\n   Train images: {train_count}")
print(f"   Val images: {val_count}")

## 🚀 Step 3: Install Ultralytics & Train

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 Extra Large (highest accuracy)
model = YOLO('yolov8x.pt')
print(f"\n📦 Model: YOLOv8x (Extra Large)")
print(f"   Parameters: ~68.2M")
print(f"   Image size: 1280px")
print(f"   This will take ~3-6 hours on A100")

In [ ]:
# ============================================================
# 🔥 TRAIN YOLOv8 EXTRA LARGE (MAX ACCURACY)
# ============================================================
# This is the main training cell. It will run for 100 epochs
# with early stopping (patience=25).
#
# The A100 GPU has 40GB VRAM, which allows a batch size of 16 
# even with the massive Extra Large model at 1280px.
# ============================================================

results = model.train(
    data=COLAB_YAML,
    epochs=100,
    imgsz=1280,           # High res for small traffic signs
    batch=16,             # 40GB VRAM will handle batch 16 easily
    project='/content/drive/MyDrive/traffic-sign-training',
    name='yolov8x_1280',
    device=0,
    workers=4,
    patience=25,          # Early stopping patience
    save=True,
    save_period=5,        # Save checkpoint every 5 epochs
    verbose=True,
    # Augmentation (tuned for traffic signs)
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,           # No vertical flip for traffic signs
    fliplr=0.3,
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    cos_lr=True,          # Cosine learning rate schedule
)

## 📊 Step 4: View Results & Download Weights

In [ ]:
# Validation results
print("📊 Validation Results:")
metrics = model.val()
print(f"   mAP50:    {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall:    {metrics.box.mr:.4f}")

In [ ]:
import shutil
from pathlib import Path

# Find best weights
run_dir = Path(results.save_dir)
best_weights = run_dir / 'weights' / 'best.pt'

if best_weights.exists():
    # Copy to easy-to-find location in Drive
    dest = Path(PROJECT_DIR) / 'yolo_best_x.pt'
    shutil.copy2(best_weights, dest)
    print(f"\n💾 Best weights saved to Google Drive:")
    print(f"   {dest}")
    print(f"   Size: {dest.stat().st_size / 1024 / 1024:.1f} MB")
    print(f"\n📥 Download this file and place it in your local project at:")
    print(f"   ml/models/yolo_best.pt")
else:
    print("❌ Best weights not found. Check training output.")

In [ ]:
# Optional: Download directly to your machine
from google.colab import files
if best_weights.exists():
    files.download(str(best_weights))